
# BTVN: Mô hình DL — RNN, Encoder–Decoder, GAN, Transformer

**Tác giả:** ChatGPT • **Ngày tạo:** 2025-10-18 15:45  
Chạy được trên CPU, không cần Internet. Thư viện: `torch`, `scikit-learn`, `matplotlib`.


## 0) Cài đặt & import

In [ ]:

# !pip install -U torch torchvision scikit-learn matplotlib

import math, random, os
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", device)


---
# 5) RNN (LSTM) — Dự báo chuỗi thời gian

In [ ]:

# Tạo chuỗi sin có nhiễu và tạo mẫu cửa sổ trượt
T = 600
x = np.arange(T)
y = np.sin(0.03*x) + 0.5*np.sin(0.05*x + 0.5) + 0.1*np.random.randn(T)
win = 30
X, Y = [], []
for i in range(T - win):
    X.append(y[i:i+win])
    Y.append(y[i+win])
X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, shuffle=False)
X_train = torch.tensor(X_train).unsqueeze(-1)
X_test  = torch.tensor(X_test).unsqueeze(-1)
y_train = torch.tensor(y_train).unsqueeze(-1)
y_test  = torch.tensor(y_test).unsqueeze(-1)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

class LSTMForecaster(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out,_ = self.lstm(x)
        return self.fc(out[:,-1,:])

model = LSTMForecaster().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
crit = nn.MSELoss()

for ep in range(1, 11):
    model.train(); tl=0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward(); opt.step()
        tl += loss.item()*xb.size(0)
    tl /= len(train_loader.dataset)
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device)).cpu().numpy().flatten()
    mse = np.mean((preds - y_test.numpy().flatten())**2)
    print(f"Epoch {ep:02d}: train_mse={tl:.4f} | test_mse={mse:.4f}")

plt.figure(figsize=(8,3))
plt.plot(range(len(y)), y, label="Thực")
plt.plot(range(len(y)-len(preds), len(y)), preds, label="Dự báo")
plt.legend(); plt.title("LSTM dự báo chuỗi"); plt.tight_layout(); plt.show()


---
# 6) Encoder–Decoder (Seq2Seq) — Đảo chuỗi (toy)

In [ ]:

import string
alphabet = string.ascii_lowercase + " "
vocab = sorted(set(alphabet))
stoi = {ch:i+1 for i,ch in enumerate(vocab)}; itos = {i:s for s,i in stoi.items()}
PAD=0; max_len=16

def enc(s):
    s=s.lower()[:max_len]
    ids=[stoi.get(ch,0) for ch in s]; ids += [PAD]*(max_len-len(ids))
    return ids
def dec(ids): return "".join(itos.get(i,"?") for i in ids if i!=PAD)

N=2000; rng=np.random.default_rng(0)
X=[]; Y=[]
for _ in range(N):
    L=rng.integers(4,max_len+1)
    s="".join(rng.choice(list(vocab), size=L)); t=s[::-1]
    X.append(enc(s)); Y.append(enc(t))
X=torch.tensor(X); Y=torch.tensor(Y)
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=42)

class Seq2Seq(nn.Module):
    def __init__(self, V, d=128):
        super().__init__()
        self.emb=nn.Embedding(V+1, d, padding_idx=PAD)
        self.enc=nn.LSTM(d,d,batch_first=True)
        self.dec=nn.LSTM(d,d,batch_first=True)
        self.fc=nn.Linear(d,V+1)
    def forward(self, src, tgt):
        _,(h,c)=self.enc(self.emb(src))
        out,_=self.dec(self.emb(tgt), (h,c))
        return self.fc(out)

model=Seq2Seq(len(vocab)).to(device)
opt=optim.Adam(model.parameters(), lr=1e-3)
crit=nn.CrossEntropyLoss(ignore_index=PAD)
B=64
loader=DataLoader(TensorDataset(Xtr,Ytr), batch_size=B, shuffle=True)

for ep in range(1, 9):
    model.train(); loss_sum=0.0
    for xb,yb in loader:
        xb,yb=xb.to(device), yb.to(device)
        inp=torch.roll(yb,1,1); inp[:,0]=PAD
        opt.zero_grad()
        logits=model(xb, inp)
        loss=crit(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        loss.backward(); opt.step()
        loss_sum += loss.item()*xb.size(0)
    print(f"Epoch {ep:02d}: train_loss={loss_sum/len(Xtr):.3f}")

@torch.no_grad()
def infer(s):
    src=torch.tensor([enc(s)], dtype=torch.long).to(device)
    _,(h,c)=model.enc(model.emb(src))
    prev=torch.tensor([[PAD]], dtype=torch.long).to(device)
    out=[]
    for _ in range(max_len):
        dec_out,(h,c)=model.dec(model.emb(prev), (h,c))
        nxt=model.fc(dec_out[:,-1,:]).argmax(-1)
        out.append(nxt.item()); prev=nxt.unsqueeze(1)
    return dec(out)

for s in ["hello world", "abc xyz", "vietnam", "data science"]:
    print(s, "->", infer(s))


---
# 7) GAN — Sinh dữ liệu two-moons (2D)

In [ ]:

X,_=make_moons(2000, noise=0.05, random_state=0)
X=X.astype(np.float32)
class G(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(
        nn.Linear(2,32), nn.ReLU(), nn.Linear(32,32), nn.ReLU(), nn.Linear(32,2))
    def forward(self,z): return self.net(z)
class D(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(
        nn.Linear(2,32), nn.ReLU(), nn.Linear(32,32), nn.ReLU(), nn.Linear(32,1), nn.Sigmoid())
    def forward(self,x): return self.net(x)

Gm, Dm = G().to(device), D().to(device)
optG=optim.Adam(Gm.parameters(), lr=1e-3); optD=optim.Adam(Dm.parameters(), lr=1e-3)
bce=nn.BCELoss(); data=torch.tensor(X, device=device); B=128

for ep in range(1, 401):
    idx=torch.randperm(data.size(0), device=device)
    for i in range(0, len(idx), B):
        real=data[idx[i:i+B]]
        z=torch.randn(real.size(0),2, device=device)
        # D
        optD.zero_grad()
        lossD = bce(Dm(real), torch.ones(real.size(0),1, device=device)) +                 bce(Dm(Gm(z).detach()), torch.zeros(real.size(0),1, device=device))
        lossD.backward(); optD.step()
        # G
        optG.zero_grad()
        lossG = bce(Dm(Gm(z)), torch.ones(real.size(0),1, device=device))
        lossG.backward(); optG.step()
    if ep%50==0: print(f"Epoch {ep}: lossD={lossD.item():.3f} lossG={lossG.item():.3f}")

with torch.no_grad():
    gen=Gm(torch.randn(2000,2, device=device)).cpu().numpy()

plt.figure(figsize=(6,3))
plt.subplot(1,2,1); plt.scatter(X[:,0],X[:,1],s=4); plt.title("Thật")
plt.subplot(1,2,2); plt.scatter(gen[:,0],gen[:,1],s=4); plt.title("GAN sinh")
plt.tight_layout(); plt.show()


---
# 8) Transformer — Ngôn ngữ ký tự (Tiny LM)

In [ ]:

text=("deep learning changes the way we build intelligent systems. "
      "transformers are powerful for sequence modeling. "
      "rnn and cnn are also useful for many tasks. "
      "this is a tiny corpus for a tiny demo.")
chars=sorted(list(set(text))); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for ch,i in stoi.items()}
vocab_size=len(chars)
def enc(s): return [stoi[c] for c in s]
def dec(ids): return "".join(itos[i] for i in ids)
data=torch.tensor(enc(text), dtype=torch.long)
block=32
def get_batch(B=64):
    ix=torch.randint(len(data)-block-1, (B,))
    x=torch.stack([data[i:i+block] for i in ix])
    y=torch.stack([data[i+1:i+block+1] for i in ix])
    return x.to(device), y.to(device)

class TinyTF(nn.Module):
    def __init__(self, d=64, heads=4, layers=2):
        super().__init__()
        self.tok=nn.Embedding(vocab_size,d)
        self.pos=nn.Embedding(block,d)
        enc_layer=nn.TransformerEncoderLayer(d_model=d, nhead=heads, batch_first=True)
        self.enc=nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.lm=nn.Linear(d, vocab_size)
    def forward(self, idx):
        B,T=idx.size()
        x=self.tok(idx)+self.pos(torch.arange(T, device=idx.device)).unsqueeze(0).expand(B,T,-1)
        x=self.enc(x)
        return self.lm(x)

model=TinyTF().to(device); opt=optim.Adam(model.parameters(), lr=3e-3); lossf=nn.CrossEntropyLoss()
for step in range(800):
    x,y=get_batch(64)
    logits=model(x); loss=lossf(logits.view(-1, vocab_size), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (step+1)%200==0: print(f"step {step+1}: loss={loss.item():.3f}")

@torch.no_grad()
def generate(start="trans", steps=150):
    idx=torch.tensor([enc(start)], dtype=torch.long).to(device)
    for _ in range(steps):
        if idx.size(1)>block: idx=idx[:,-block:]
        logits=model(idx)[:,-1,:]
        probs=torch.softmax(logits,dim=-1)
        nxt=torch.multinomial(probs,1)
        idx=torch.cat([idx,nxt], dim=1)
    return dec(idx[0].tolist())

print("\nSinh văn bản:")
print(generate("trans", 150))
